In [4]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent

# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

In [5]:
from typing import List, Dict
from langchain.tools import tool

@tool
def send_email_tool(to: str, subject: str, body: str) -> str:
    """지정한 주소로 이메일을 보내는 도구(프로토타입)."""
    return f"Email sent. to={to}, subject={subject}"

@tool
def read_email_tool(limit: int = 3) -> List[Dict[str, str]]:
    """최근 받은 이메일을 조회하는 도구(프로토타입)."""
    return [{"from": "hr@example.com", "subject": "정책 안내", "body": "..."}][:limit]

In [6]:
from langchain.agents.middleware import LLMToolEmulator


LLMToolEmulator

agent = create_agent(
    model=model,
    tools=[send_email_tool, read_email_tool],
    # tool emulator: 실제 도구 호출 대신 LLM으로 시뮬레이션, 즉, 도구가 만든 결과를 모킹하는것. gemini-2.5-flash 모델 사용
    middleware=[LLMToolEmulator(model="gpt-4o-mini")],
    )

# # 이메일 읽기 예시
result = agent.invoke({"messages": [
  {"role": "user", "content": "최근 도착한 이메일을 읽고 답장해줘"}]},
  {"configurable": {"thread_id": 1}})

print("=== 이메일 읽고 쓰기 결과 ===")
print(result)
print(result['messages'][-1].content)



=== 이메일 읽고 쓰기 결과 ===
{'messages': [HumanMessage(content='최근 도착한 이메일을 읽고 답장해줘', additional_kwargs={}, response_metadata={}, id='466b0249-bedb-43e6-ba48-625933d4c530'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 110, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DLfZWxGMvsAHOIdsc1G3ZDBJWgp0Y', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d0e0a-7ef7-7251-a88e-4809df3ed64e-0', tool_calls=[{'name': 'read_email_tool', 'args': {'limit': 3}, 'id': 'call_A2JppPe7UDt8AoSpyV06YIO7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1